In [1]:
%%writefile ecommerce_app.py
"""EduCommerce: in-memory application for practicing checkout and automated testing."""
from dataclasses import dataclass, field
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional

D = lambda x: Decimal(str(x))
def money(x): return D(x).quantize(D('1'), rounding=ROUND_HALF_UP)

@dataclass
class Product:
    sku: str; title: str; price: Decimal; stock: int; active: bool = True

@dataclass
class Customer:
    customer_id: str; authenticated: bool = True; used_codes: set = field(default_factory=set)

@dataclass
class CartLine:
    sku: str; quantity: int

@dataclass
class Voucher:
    code: str; percent: int; min_subtotal: Decimal; cap: Decimal; active: bool = True

@dataclass
class Order:
    order_id: str; customer_id: str; total: Decimal; status: str; lines: list
    discount: Decimal; shipping: Decimal; payment_method: str

class Catalog:
    def __init__(self):
        self.products = {
            'A': Product('A', 'Wireless Mouse', D(200000), 20),
            'B': Product('B', 'Mechanical Keyboard', D(300000), 15),
            'C': Product('C', 'USB-C Hub', D(150000), 12)
        }
    def get(self, sku):
        p = self.products.get(sku)
        if not p or not p.active: raise ValueError('PRODUCT_UNAVAILABLE')
        return p

class Cart:
    def __init__(self, catalog): self.catalog = catalog; self.lines = []
    def add(self, sku, quantity):
        self.catalog.get(sku)
        if not isinstance(quantity, int) or quantity < 1: raise ValueError('INVALID_QUANTITY')
        existing = next((x for x in self.lines if x.sku == sku), None)
        if existing: existing.quantity += quantity
        else: self.lines.append(CartLine(sku, quantity))
    def subtotal(self): return sum((self.catalog.get(x.sku).price * x.quantity for x in self.lines), D(0))

class Inventory:
    def __init__(self, catalog): self.catalog = catalog
    def reserve(self, lines):
        for line in lines:
            if self.catalog.get(line.sku).stock < line.quantity: raise ValueError('OUT_OF_STOCK')
        for line in lines: self.catalog.get(line.sku).stock -= line.quantity
    def release(self, lines):
        for line in lines: self.catalog.get(line.sku).stock += line.quantity

class Promotions:
    def __init__(self): self.codes = {'SAVE10': Voucher('SAVE10', 10, D(500000), D(100000))}
    def discount(self, code, subtotal, customer):
        if not code: return D(0)
        v = self.codes.get(code)
        if not v or not v.active: raise ValueError('INVALID_VOUCHER')
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if code in customer.used_codes: raise ValueError('VOUCHER_USED')
        if subtotal < v.min_subtotal: raise ValueError('MIN_SUBTOTAL_NOT_MET')
        return min(money(subtotal * v.percent / D(100)), v.cap)

class Shipping:
    def calculate(self, subtotal, area):
        if area not in ('inner', 'outer'): raise ValueError('INVALID_AREA')
        # Lỗi tiềm ẩn trong code gốc Instruction: dùng '>' thay vì '>=' ở ngưỡng 500000
        return D(0) if area == 'inner' and subtotal > D(500000) else (D(30000) if area == 'inner' else D(50000))

class Payments:
    def __init__(self): self.charges = []
    def collect(self, amount, method, approved, key):
        if method == 'COD': return 'PENDING_COD'
        if method != 'ONLINE': raise ValueError('INVALID_PAYMENT_METHOD')
        if not approved: raise ValueError('PAYMENT_DECLINED')
        self.charges.append({'key': key, 'amount': amount})
        return 'PAID'

class Checkout:
    def __init__(self, catalog=None):
        self.catalog = catalog or Catalog(); self.inventory = Inventory(self.catalog)
        self.promotions = Promotions(); self.shipping = Shipping(); self.payments = Payments()
        self.orders = {}; self.by_key = {}; self.counter = 0

    def place(self, customer, cart, area='inner', voucher=None, method='COD', approved=True, key='REQ-001'):
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if not cart.lines: raise ValueError('EMPTY_CART')
        subtotal = cart.subtotal()
        discount = self.promotions.discount(voucher, subtotal, customer)
        shipping = self.shipping.calculate(subtotal, area)
        total = max(D(0), subtotal - discount + shipping)
        # Reserve stock during the checkout transaction.
        self.inventory.reserve(cart.lines)
        if key in self.by_key: return self.orders[self.by_key[key]]
        # Process the online payment outcome.
        payment_state = self.payments.collect(total, method, approved, key)
        self.counter += 1
        order = Order(f'ORD-{self.counter:04d}', customer.customer_id, total, payment_state,
                      [(x.sku, x.quantity) for x in cart.lines], discount, shipping, method)
        self.orders[order.order_id] = order; self.by_key[key] = order.order_id
        if voucher: customer.used_codes.add(voucher)
        return order

def fixture(subtotal_case='600k'):
    service = Checkout(); customer = Customer('U_NEW'); cart = Cart(service.catalog)
    if subtotal_case == 'empty':
        pass
    elif subtotal_case == 'inactive_A':
        service.catalog.products['A'].active = False
    elif subtotal_case == 'unauth_600k':
        customer.authenticated = False
        cart.add('A', 3)
    elif subtotal_case == 'used_save10_600k':
        customer.used_codes.add('SAVE10')
        cart.add('A', 3)
    elif subtotal_case == '500k':
        cart.add('A', 1); cart.add('B', 1)
    elif subtotal_case == '500001':
        cart.add('A', 1); cart.add('B', 1); service.catalog.products['B'].price = D(300001)
    elif subtotal_case == '499999':
        cart.add('A', 1); cart.add('B', 1); service.catalog.products['B'].price = D(299999)
    elif subtotal_case == '1000k':
        cart.add('A', 5)
    elif subtotal_case == '1500k':
        cart.add('B', 5)
    elif subtotal_case == '600k':
        cart.add('A', 3)
    elif subtotal_case == '300k':
        cart.add('B', 1)
    else:
        raise ValueError('UNKNOWN_FIXTURE')
    return service, customer, cart

Overwriting ecommerce_app.py


In [2]:
%%writefile TestCases.csv
test_case_id,requirement_id,condition_id,scenario_id,title,priority,precondition,fixture_key,input_json,expected_type,expected_json,execution_status,actual_result,error_message,evidence_reference
TC-CAT-001,REQ-01,COND-CAT-01,TS-CAT-01,Truy van san pham A dang active thanh cong,P1,Catalog khoi tao mac dinh SKU A active=True,empty,"{""action"":""catalog_get"",""sku"":""A""}",VALUE,"{""sku"":""A"",""price"":""200000"",""active"":true}",,,,Catalog.get
TC-CAT-002,REQ-01,COND-CAT-02,TS-CAT-02,Tu choi truy van san pham A khi active=False,P0,SKU A bi danh dau active=False,inactive_A,"{""action"":""catalog_get"",""sku"":""A""}",EXCEPTION,"{""error"":""PRODUCT_UNAVAILABLE""}",,,,Catalog.get
TC-CAT-003,REQ-01,COND-CAT-02,TS-CAT-02,Tu choi truy van SKU Z khong ton tai trong Catalog,P1,Catalog khoi tao mac dinh khong co Z,empty,"{""action"":""catalog_get"",""sku"":""Z""}",EXCEPTION,"{""error"":""PRODUCT_UNAVAILABLE""}",,,,Catalog.get
TC-CRT-001,REQ-02,COND-CRT-01,TS-CRT-01,Them SKU A hai lan vao gio cong don so luong va tinh subtotal,P0,Gio hang rong SKU A active,empty,"{""action"":""cart_add_seq"",""items"":[[""A"",2],[""A"",3]]}",STATE,"{""lines_count"":1,""total_qty"":5,""subtotal"":""1000000""}",,,,Cart.add
TC-CRT-002,REQ-02,COND-CRT-02,TS-CRT-02,Tu choi them san pham voi so luong bang 0 (Boundary),P1,Gio hang rong SKU A active,empty,"{""action"":""cart_add_seq"",""items"":[[""A"",0]]}",EXCEPTION,"{""error"":""INVALID_QUANTITY""}",,,,Cart.add
TC-CRT-003,REQ-02,COND-CRT-02,TS-CRT-02,Tu choi them san pham voi so luong am (-2),P1,Gio hang rong SKU A active,empty,"{""action"":""cart_add_seq"",""items"":[[""A"",-2]]}",EXCEPTION,"{""error"":""INVALID_QUANTITY""}",,,,Cart.add
TC-PRO-001,REQ-07,COND-PRO-01,TS-PRO-01,Tu choi ma voucher khong ton tai SAVE99,P1,User dang nhap gio hang 600k,600k,"{""action"":""checkout"",""voucher"":""SAVE99"",""method"":""COD""}",EXCEPTION,"{""error"":""INVALID_VOUCHER"",""voucher_used"":false}",,,,Promotions.discount
TC-PRO-002,REQ-07,COND-PRO-01,TS-PRO-01,Tu choi ma SAVE10 khi khach hang da su dung truoc do,P0,User dang nhap used_codes co SAVE10 gio 600k,used_save10_600k,"{""action"":""checkout"",""voucher"":""SAVE10"",""method"":""COD""}",EXCEPTION,"{""error"":""VOUCHER_USED"",""voucher_used"":true}",,,,Promotions.discount
TC-PRO-003,REQ-07,COND-PRO-02,TS-PRO-02,Khong danh dau SAVE10 la used khi thanh toan ONLINE bi declined,P0,User dang nhap chua dung SAVE10 gio 600k,600k,"{""action"":""checkout"",""voucher"":""SAVE10"",""method"":""ONLINE"",""approved"":false}",EXCEPTION,"{""error"":""PAYMENT_DECLINED"",""voucher_used"":false}",,,,Checkout.place
TC-SHP-001,REQ-09,COND-SHP-01,TS-SHP-01,Tinh phi ship noi thanh 30k khi subtotal duoi nguong (499.999 VND),P1,Gio hang subtotal 499999 area inner,499999,"{""action"":""shipping_calc"",""area"":""inner""}",VALUE,"{""shipping_fee"":""30000""}",,,,Shipping.calculate
TC-SHP-002,REQ-09,COND-SHP-01,TS-SHP-01,Mien phi ship noi thanh (0 VND) tai dung nguong subtotal 500.000 VND,P0,Gio hang subtotal 500000 area inner,500k,"{""action"":""shipping_calc"",""area"":""inner""}",VALUE,"{""shipping_fee"":""0""}",,,,Shipping.calculate
TC-SHP-003,REQ-09,COND-SHP-02,TS-SHP-02,Tinh phi ship ngoai thanh 50k ke ca khi subtotal dat 600.000 VND,P1,Gio hang subtotal 600000 area outer,600k,"{""action"":""shipping_calc"",""area"":""outer""}",VALUE,"{""shipping_fee"":""50000""}",,,,Shipping.calculate
TC-PAY-001,REQ-10,COND-PAY-01,TS-PAY-01,Tao don hang voi trang thai PENDING_COD khi thanh toan COD,P1,User dang nhap gio 600k method COD,600k,"{""action"":""checkout"",""method"":""COD"",""area"":""inner""}",STATE,"{""status"":""PENDING_COD"",""total"":""600000"",""orders_count"":1}",,,,Checkout.place
TC-PAY-002,REQ-10,COND-PAY-02,TS-PAY-02,Tao don hang PAID va ghi nhan charge khi ONLINE approved=True,P0,User dang nhap gio 600k method ONLINE approved=True,600k,"{""action"":""checkout"",""method"":""ONLINE"",""approved"":true}",STATE,"{""status"":""PAID"",""charges_count"":1,""orders_count"":1}",,,,Checkout.place
TC-PAY-003,REQ-10,COND-PAY-02,TS-PAY-02,Tu choi tao don khi ONLINE approved=False,P0,User dang nhap gio 600k method ONLINE approved=False,600k,"{""action"":""checkout"",""method"":""ONLINE"",""approved"":false}",EXCEPTION,"{""error"":""PAYMENT_DECLINED"",""orders_count"":0}",,,,Checkout.place
TC-INV-001,REQ-15,COND-INV-01,TS-INV-01,Tu choi toan bo don va giu nguyen stock A khi SKU B thieu ton kho,P0,Gio hang gom A=2 (du) va B=100 (thieu),empty,"{""action"":""atomicity_check"",""items"":[[""A"",2],[""B"",100]]}",EXCEPTION,"{""error"":""OUT_OF_STOCK"",""stock_A"":20,""stock_B"":15,""orders_count"":0}",,,,Inventory.reserve
TC-INV-002,REQ-15,COND-INV-02,TS-INV-02,Tru ton kho chinh xac cho ca A va B khi ca 2 SKU deu du ton kho,P1,Gio hang gom A=2 va B=3 (deu du),empty,"{""action"":""atomicity_check"",""items"":[[""A"",2],[""B"",3]]}",STATE,"{""status"":""PENDING_COD"",""stock_A"":18,""stock_B"":12,""orders_count"":1}",,,,Inventory.reserve
TC-SAF-001,REQ-16,COND-SAF-01,TS-SAF-01,Tu choi phuong thuc thanh toan WALLET va khong lam mat ton kho,P0,User dang nhap gio 600k (A=3) method WALLET,600k,"{""action"":""payment_safety"",""method"":""WALLET""}",EXCEPTION,"{""error"":""INVALID_PAYMENT_METHOD"",""stock_A"":20,""orders_count"":0}",,,,Checkout.place

Overwriting TestCases.csv


In [3]:
%%writefile test_runner.py
"""Data-driven test runner for STLab02 Requirement Execution."""
from pathlib import Path
import csv, datetime, html, json, time
from collections import Counter
import pandas as pd
from ecommerce_app import fixture

ROOT = Path(__file__).resolve().parent
REQUIRED_COLUMNS = [
    'test_case_id', 'requirement_id', 'condition_id', 'scenario_id', 'title',
    'priority', 'precondition', 'fixture_key', 'input_json', 'expected_type',
    'expected_json', 'execution_status', 'actual_result', 'error_message', 'evidence_reference'
]

def execute_case(row):
    start_time = time.perf_counter()
    tc_id = row.get('test_case_id', 'UNKNOWN')
    try:
        app, user, cart = fixture(row['fixture_key'])
        inp = json.loads(row['input_json'])
        exp = json.loads(row['expected_json'])
    except Exception as setup_err:
        elapsed = round((time.perf_counter() - start_time) * 1000, 2)
        return {
            **row,
            'execution_status': 'BLOCKED',
            'actual_result': json.dumps({'setup_error': str(setup_err)}),
            'error_message': f'SETUP/PARSE ERROR: {setup_err}',
            'execution_time_ms': elapsed,
            'executed_at': datetime.datetime.now(datetime.timezone.utc).isoformat()
        }

    action = inp.get('action')
    actual_data = {}
    error_msg = ''
    passed = False

    try:
        if action == 'catalog_get':
            prod = app.catalog.get(inp['sku'])
            actual_data = {'sku': prod.sku, 'price': str(prod.price), 'active': prod.active}

        elif action == 'cart_add_seq':
            for sku, qty in inp['items']:
                cart.add(sku, qty)
            total_qty = sum(x.quantity for x in cart.lines)
            actual_data = {
                'lines_count': len(cart.lines),
                'total_qty': total_qty,
                'subtotal': str(cart.subtotal())
            }

        elif action == 'shipping_calc':
            fee = app.shipping.calculate(cart.subtotal(), inp['area'])
            actual_data = {'shipping_fee': str(fee)}

        elif action == 'checkout':
            order = app.place(
                user, cart,
                area=inp.get('area', 'inner'),
                voucher=inp.get('voucher'),
                method=inp.get('method', 'COD'),
                approved=inp.get('approved', True),
                key=tc_id
            )
            actual_data = {
                'status': order.status,
                'total': str(order.total),
                'charges_count': len(app.payments.charges),
                'orders_count': len(app.orders),
                'voucher_used': ('SAVE10' in user.used_codes)
            }

        elif action == 'atomicity_check':
            for sku, qty in inp['items']:
                cart.add(sku, qty)
            order = app.place(user, cart, method='COD', key=tc_id)
            actual_data = {
                'status': order.status,
                'stock_A': app.catalog.products['A'].stock,
                'stock_B': app.catalog.products['B'].stock,
                'orders_count': len(app.orders)
            }

        elif action == 'payment_safety':
            order = app.place(user, cart, method=inp['method'], key=tc_id)
            actual_data = {
                'status': order.status,
                'stock_A': app.catalog.products['A'].stock,
                'orders_count': len(app.orders)
            }
        else:
            raise RuntimeError(f'UNKNOWN_ACTION: {action}')

    except ValueError as ve:
        error_msg = str(ve)
        actual_data = {
            'error': error_msg,
            'stock_A': app.catalog.products['A'].stock,
            'stock_B': app.catalog.products['B'].stock,
            'orders_count': len(app.orders),
            'voucher_used': ('SAVE10' in user.used_codes)
        }
    except Exception as unexpected_err:
        elapsed = round((time.perf_counter() - start_time) * 1000, 2)
        return {
            **row,
            'execution_status': 'BLOCKED',
            'actual_result': json.dumps({'unexpected_error': str(unexpected_err)}),
            'error_message': str(unexpected_err),
            'execution_time_ms': elapsed,
            'executed_at': datetime.datetime.now(datetime.timezone.utc).isoformat()
        }

    # Compare only keys specified in expected_json (Oracle verification)
    filtered_actual = {k: actual_data.get(k) for k in exp.keys()}
    if row['expected_type'] == 'EXCEPTION':
        passed = (error_msg != '') and (filtered_actual == exp)
    else:
        passed = (error_msg == '') and (filtered_actual == exp)

    elapsed = round((time.perf_counter() - start_time) * 1000, 2)
    return {
        **row,
        'execution_status': 'PASS' if passed else 'FAIL',
        'actual_result': json.dumps(filtered_actual),
        'error_message': error_msg,
        'execution_time_ms': elapsed,
        'executed_at': datetime.datetime.now(datetime.timezone.utc).isoformat()
    }

def run_suite(csv_path=None, out_dir=None):
    csv_path = Path(csv_path or ROOT / 'TestCases.csv')
    output = Path(out_dir or ROOT / 'reports')
    output.mkdir(exist_ok=True, parents=True)

    with csv_path.open(newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        missing_cols = [c for c in REQUIRED_COLUMNS if c not in (reader.fieldnames or [])]
        if missing_cols:
            raise ValueError(f'MISSING_REQUIRED_COLUMNS: {missing_cols}')
        testcases = list(reader)

    ids = [x['test_case_id'] for x in testcases]
    if len(ids) != len(set(ids)):
        raise ValueError('DUPLICATE_TEST_CASE_IDS')

    results = [execute_case(x) for x in testcases]
    df = pd.DataFrame(results)

    summary = df.groupby(['requirement_id', 'execution_status']).size().unstack(fill_value=0).reset_index()
    for col in ['PASS', 'FAIL', 'BLOCKED']:
        if col not in summary.columns:
            summary[col] = 0
    summary['Total'] = summary['PASS'] + summary['FAIL'] + summary['BLOCKED']
    summary['Pass_Rate (%)'] = (summary['PASS'] / summary['Total'] * 100).round(1)

    counts = Counter(df['execution_status'])
    totals = pd.DataFrame([{
        'total_cases': len(df),
        'passed': counts.get('PASS', 0),
        'failed': counts.get('FAIL', 0),
        'blocked': counts.get('BLOCKED', 0),
        'pass_rate_percent': round(100 * counts.get('PASS', 0) / len(df), 1) if len(df) else 0
    }])

    df.to_csv(output / 'test_execution.csv', index=False, encoding='utf-8-sig')
    with pd.ExcelWriter(output / 'test_report.xlsx', engine='openpyxl') as xls:
        df.to_excel(xls, index=False, sheet_name='Execution')
        summary.to_excel(xls, index=False, sheet_name='Traceability')
        totals.to_excel(xls, index=False, sheet_name='Summary')

    rows_html = ''.join(
        '<tr>' + ''.join(
            f'<td>{html.escape(str(r.get(k, "")))}</td>'
            for k in ['test_case_id', 'requirement_id', 'condition_id', 'scenario_id', 'expected_json', 'actual_result', 'execution_status', 'execution_time_ms']
        ) + '</tr>'
        for r in results
    )
    page = f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>EduCommerce Test Report</title>
    <style>body{{font:14px Arial;margin:30px;color:#14233c}}h1{{color:#17385f}}table{{border-collapse:collapse;width:100%}}
    td,th{{border:1px solid #cbd5e1;padding:8px;text-align:left}}th{{background:#16385e;color:white}}
    tr:nth-child(even){{background:#f1f5f9}}.metric{{display:inline-block;margin:10px;padding:14px;border-radius:10px;background:#edf3f9;font-size:20px;font-weight:bold}}</style>
    </head><body><h1>EduCommerce Automated Test Execution Report</h1>
    <div class="metric">Total: {len(df)}</div>
    <div class="metric" style="color:#16a34a">PASS: {counts.get("PASS", 0)}</div>
    <div class="metric" style="color:#dc2626">FAIL: {counts.get("FAIL", 0)}</div>
    <div class="metric" style="color:#d97706">BLOCKED: {counts.get("BLOCKED", 0)}</div>
    <div class="metric">Pass Rate: {totals['pass_rate_percent'].iloc[0]}%</div>
    <h2>Traceability & Execution Details</h2>
    <table><tr><th>Case ID</th><th>REQ</th><th>Condition</th><th>Scenario</th><th>Expected</th><th>Actual</th><th>Status</th><th>Time (ms)</th></tr>
    {rows_html}</table></body></html>"""
    (output / 'test_dashboard.html').write_text(page, encoding='utf-8')

    print(f"Executed {len(df)} | PASS {counts.get('PASS',0)} | FAIL {counts.get('FAIL',0)} | BLOCKED {counts.get('BLOCKED',0)}")
    print("\n--- SUMMARY BY REQUIREMENT ---")
    print(summary.to_string(index=False))
    print("\n--- FAILED TEST CASES ---")
    print(df.loc[df['execution_status'] == 'FAIL', ['test_case_id', 'requirement_id', 'expected_json', 'actual_result', 'error_message']].to_string(index=False))
    return df, output

if __name__ == '__main__':
    run_suite()

Overwriting test_runner.py


In [4]:
!pip -q install pandas openpyxl matplotlib gradio
!python test_runner.py

Executed 18 | PASS 16 | FAIL 2 | BLOCKED 0

--- SUMMARY BY REQUIREMENT ---
requirement_id  FAIL  PASS  BLOCKED  Total  Pass_Rate (%)
        REQ-01     0     3        0      3          100.0
        REQ-02     0     3        0      3          100.0
        REQ-07     0     3        0      3          100.0
        REQ-09     1     2        0      3           66.7
        REQ-10     0     3        0      3          100.0
        REQ-15     0     2        0      2          100.0
        REQ-16     1     0        0      1            0.0

--- FAILED TEST CASES ---
test_case_id requirement_id                                                    expected_json                                                         actual_result          error_message
  TC-SHP-002         REQ-09                                             {"shipping_fee":"0"}                                             {"shipping_fee": "30000"}                       
  TC-SAF-001         REQ-16 {"error":"INVALID_PAYMENT_METHOD",

In [5]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd

report_df = pd.read_csv("reports/test_execution.csv").fillna("")

def dashboard_view(req="ALL", status="ALL"):
    df = report_df.copy()
    if req != "ALL": df = df[df["requirement_id"] == req]
    if status != "ALL": df = df[df["execution_status"] == status]
    summary = df["execution_status"].value_counts().reindex(["PASS", "FAIL", "BLOCKED"], fill_value=0)

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(summary.index, summary.values, color=["#16a34a", "#dc2626", "#d97706"])
    ax.bar_label(bars, padding=3)
    ax.set_title("EduCommerce Test Execution Results")
    ax.set_ylabel("Number of Test Cases")
    ax.set_ylim(0, max(2, int(summary.max()) + 3))
    fig.tight_layout()

    cols = ["test_case_id", "requirement_id", "scenario_id", "expected_json", "actual_result", "execution_status", "execution_time_ms"]
    return fig, df[cols], f"**Executed:** {len(df)} | **PASS:** {summary['PASS']} | **FAIL:** {summary['FAIL']} | **BLOCKED:** {summary['BLOCKED']}"

with gr.Blocks(title="EduCommerce Test Dashboard") as demo:
    gr.Markdown("# EduCommerce Requirement Traceability & Test Results Dashboard")
    with gr.Row():
        req_filter = gr.Dropdown(["ALL"] + sorted(report_df.requirement_id.unique().tolist()), value="ALL", label="Requirement ID")
        status_filter = gr.Dropdown(["ALL", "PASS", "FAIL", "BLOCKED"], value="ALL", label="Execution Status")
    status_line = gr.Markdown()
    with gr.Row(equal_height=True):
        with gr.Column(scale=1, min_width=300):
            chart = gr.Plot(label="Status Distribution")
        with gr.Column(scale=2, min_width=450):
            result_table = gr.Dataframe(label="Test Execution Details", interactive=False, wrap=True)
    gr.File(value="reports/test_report.xlsx", label="Download Excel Report")

    req_filter.change(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])
    status_filter.change(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])
    demo.load(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3fd48f775d878b106c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
